# Lab 04
## Made by: Santiago Villa Rodríguez

In [ ]:
from spark_utils import SparkUtils

su = SparkUtils("spark://spark-master:7077", "Lab03 - Data Cleaning and Transformation Pipeline")
su._spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/03 00:08:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
from pyspark.sql import functions as Func

base_path = "/opt/spark/work-dir/data/car_service"

agencies_df = (
    su._spark.read.option("header", True) 
    .option("inferSchema", True) 
    .csv(f"{base_path}/agencies")
    .withColumn("agency_info_json", Func.regexp_replace("agency_info", "'", '"'))
    .withColumn("agency_name", Func.get_json_object("agency_info_json", "$.agency_name"))
    .withColumn("agency_city", Func.get_json_object("agency_info_json", "$.city"))
)

brands_df = (
    su._spark.read.option("header", True)
    .option("inferSchema", True)
    .csv(f"{base_path}/brands")
    .withColumn("brand_info_json", Func.regexp_replace("brand_info", "'", '"'))
    .withColumn("brand_name", Func.get_json_object("brand_info_json", "$.brand_name"))
    .withColumn("brand_country", Func.get_json_object("brand_info_json", "$.country"))
)

cars_df = (
    su._spark.read.option("header", True)
    .option("inferSchema", True)
    .csv(f"{base_path}/cars")
    .withColumn("car_info_json", Func.regexp_replace("car_info", "'", '"'))
    .withColumn("car_name", Func.get_json_object("car_info_json", "$.car_name"))
    .withColumn("brand_id", Func.get_json_object("car_info_json", "$.brand_id").cast("int"))
    .withColumn("price_per_day", Func.get_json_object("car_info_json", "$.price_per_day").cast("int"))
)

customers_df = (
    su._spark.read.option("header", True)
    .option("inferSchema", True)
    .csv(f"{base_path}/customers")
    .withColumn("customer_info_json", Func.regexp_replace("customer_info", "'", '"'))
    .withColumn("customer_name", Func.get_json_object("customer_info_json", "$.customer_name"))
    .withColumn("customer_city", Func.get_json_object("customer_info_json", "$.city"))
    .withColumn("customer_age", Func.get_json_object("customer_info_json", "$.age").cast("int"))
)

rentals_df = (
    su._spark.read.option("header", True)
    .option("inferSchema", True)
    .csv(f"{base_path}/rentals")
    .withColumn("rental_info_json", Func.regexp_replace("rental_info", "'", '"'))
    .withColumn("car_id", Func.get_json_object("rental_info_json", "$.car_id").cast("int"))
    .withColumn("customer_id", Func.get_json_object("rental_info_json", "$.customer_id").cast("int"))
    .withColumn("agency_id", Func.get_json_object("rental_info_json", "$.agency_id").cast("int"))
)

In [12]:
agencies_df.show(5)
brands_df.show(5)


+---------+--------------------+--------------------+-------------+-------------+
|agency_id|         agency_info|    agency_info_json|  agency_name|  agency_city|
+---------+--------------------+--------------------+-------------+-------------+
|        1|{'agency_name': '...|{"agency_name": "...|  NYC Rentals|     New York|
|        2|{'agency_name': '...|{"agency_name": "...|LA Car Rental|      Londres|
|        3|{'agency_name': '...|{"agency_name": "...| Zapopan Auto|      Zapopan|
|        4|{'agency_name': '...|{"agency_name": "...|      SF Cars|San Francisco|
|        5|{'agency_name': '...|{"agency_name": "...|  Mexico Cars|  Mexico City|
+---------+--------------------+--------------------+-------------+-------------+

+--------+--------------------+--------------------+-------------+-------------+
|brand_id|          brand_info|     brand_info_json|   brand_name|brand_country|
+--------+--------------------+--------------------+-------------+-------------+
|       1|{'brand_

In [13]:
cars_df.show(5)
customers_df.show(5)
rentals_df.show(5)

+------+--------------------+--------------------+--------------------+--------+-------------+
|car_id|            car_info|       car_info_json|            car_name|brand_id|price_per_day|
+------+--------------------+--------------------+--------------------+--------+-------------+
|     1|{'car_name': 'Cha...|{"car_name": "Cha...|Chang-Fisher Model 7|       5|          139|
|     2|{'car_name': 'She...|{"car_name": "She...|Sheppard-Tucker M...|       6|           70|
|     3|{'car_name': 'Fau...|{"car_name": "Fau...|Faulkner-Howard M...|       3|           53|
|     4|{'car_name': 'Wag...|{"car_name": "Wag...|  Wagner LLC Model 1|       5|           89|
|     5|{'car_name': 'Cam...|{"car_name": "Cam...|  Campos PLC Model 4|       4|          112|
+------+--------------------+--------------------+--------------------+--------+-------------+
only showing top 5 rows
+-----------+--------------------+--------------------+--------------+-------------+------------+
|customer_id|       cus

In [22]:
result = agencies_df.join(rentals_df, on="agency_id", how="left") \
      .join(cars_df, on="car_id", how="left") \
      .join(brands_df, on="brand_id", how="left") \
      .join(customers_df, on="customer_id", how="left") \
      .select(
          Func.col("rental_id"), Func.col("car_name"), Func.col("agency_name"), Func.col("customer_name")
      ) \
      .orderBy("rental_id", ascending=False)

result.show(5, truncate=False)


+---------+------------------------------------+-------------+-------------+
|rental_id|car_name                            |agency_name  |customer_name|
+---------+------------------------------------+-------------+-------------+
|17833    |Grimes-Green Model 8                |LA Car Rental|Jill Sherman |
|17832    |Walker, Pratt and Thomas Model 9    |Zapopan Auto |Troy Bell    |
|17831    |Levy Group Model 9                  |Zapopan Auto |Lisa Baldwin |
|17830    |Alvarez-Davis Model 3               |LA Car Rental|David Walker |
|17829    |Patrick, Barrera and Collins Model 6|LA Car Rental|Blake Jones  |
+---------+------------------------------------+-------------+-------------+
only showing top 5 rows


In [ ]:
su.stop()